# US Stocks Historical Volatility Analysis - Version 1\n\n**Author:** Trade Vectors LLP\n**Date:** March 2026\n\n## Overview\nThis notebook performs historical volatility analysis on US stock market data. It calculates various volatility metrics including:\n- Historical Volatility (HV)\n- Standard Deviation\n- Average True Range (ATR)\n- Parkinson Volatility\n\n## Data Source\nHistorical stock price data is loaded from CSV files in the USStocks folder.

In [ ]:
# Import required libraries\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom datetime import datetime\nimport warnings\nwarnings.filterwarnings('ignore')\n\n# Set display options\npd.set_option('display.max_columns', None)\npd.set_option('display.width', 1000)\n\nprint('Libraries imported successfully')

In [ ]:
# Load stock data\ndef load_stock_data(ticker):\n    """Load historical stock data from CSV file"""\n    file_path = f'../USStocks/{ticker}_STK.csv'\n    df = pd.read_csv(file_path)\n    df['Date'] = pd.to_datetime(df['Date'])\n    df.set_index('Date', inplace=True)\n    df.sort_index(inplace=True)\n    return df\n\n# List of tickers to analyze\ntickers = ['NVDA', 'AAPL', 'MSFT', 'AMZN', 'IBM']\n\n# Load data for all stocks\nstock_data = {}\nfor ticker in tickers:\n    try:\n        stock_data[ticker] = load_stock_data(ticker)\n        print(f'{ticker}: Loaded {len(stock_data[ticker])} records')\n    except Exception as e:\n        print(f'Error loading {ticker}: {str(e)}')

In [ ]:
# Calculate Historical Volatility\ndef calculate_historical_volatility(df, window=20):\n    """\n    Calculate historical volatility using close-to-close method\n    Returns annualized volatility\n    """\n    log_returns = np.log(df['Close'] / df['Close'].shift(1))\n    volatility = log_returns.rolling(window=window).std() * np.sqrt(252)\n    return volatility\n\n# Calculate Parkinson Volatility\ndef calculate_parkinson_volatility(df, window=20):\n    """\n    Calculate Parkinson volatility using high-low range\n    More efficient estimator than close-to-close\n    """\n    hl_ratio = np.log(df['High'] / df['Low'])\n    parkinson = (1 / (4 * np.log(2))) * (hl_ratio ** 2)\n    volatility = np.sqrt(parkinson.rolling(window=window).mean() * 252)\n    return volatility\n\n# Calculate Average True Range (ATR)\ndef calculate_atr(df, window=14):\n    """Calculate Average True Range"""\n    high_low = df['High'] - df['Low']\n    high_close = np.abs(df['High'] - df['Close'].shift())\n    low_close = np.abs(df['Low'] - df['Close'].shift())\n    \n    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)\n    atr = true_range.rolling(window=window).mean()\n    return atr

In [ ]:
# Calculate volatility metrics for all stocks\nvolatility_results = {}\n\nfor ticker in stock_data.keys():\n    df = stock_data[ticker].copy()\n    \n    # Calculate different volatility measures\n    df['HV_20'] = calculate_historical_volatility(df, window=20)\n    df['HV_30'] = calculate_historical_volatility(df, window=30)\n    df['Parkinson_Vol'] = calculate_parkinson_volatility(df, window=20)\n    df['ATR_14'] = calculate_atr(df, window=14)\n    \n    volatility_results[ticker] = df\n    \n    print(f'{ticker} - Latest Metrics:')\n    print(f'  20-day HV: {df["HV_20"].iloc[-1]:.2%}')\n    print(f'  30-day HV: {df["HV_30"].iloc[-1]:.2%}')\n    print(f'  Parkinson Vol: {df["Parkinson_Vol"].iloc[-1]:.2%}')\n    print(f'  ATR: ${df["ATR_14"].iloc[-1]:.2f}')\n    print()

In [ ]:
# Visualization: Historical Volatility Comparison\nplt.figure(figsize=(14, 8))\n\nfor ticker in volatility_results.keys():\n    df = volatility_results[ticker]\n    plt.plot(df.index, df['HV_20'], label=ticker, linewidth=2)\n\nplt.title('20-Day Historical Volatility Comparison (Annualized)', fontsize=16, fontweight='bold')\nplt.xlabel('Date', fontsize=12)\nplt.ylabel('Volatility', fontsize=12)\nplt.legend(loc='best', fontsize=10)\nplt.grid(True, alpha=0.3)\nplt.tight_layout()\nplt.show()

In [ ]:
# Summary statistics\nsummary_data = []\n\nfor ticker in volatility_results.keys():\n    df = volatility_results[ticker]\n    \n    summary_data.append({\n        'Ticker': ticker,\n        'Avg HV (20d)': df['HV_20'].mean(),\n        'Current HV (20d)': df['HV_20'].iloc[-1],\n        'Max HV (20d)': df['HV_20'].max(),\n        'Min HV (20d)': df['HV_20'].min(),\n        'Avg ATR': df['ATR_14'].mean()\n    })\n\nsummary_df = pd.DataFrame(summary_data)\nprint('Volatility Summary Statistics')\nprint('=' * 80)\nprint(summary_df.to_string(index=False))\nprint('=' * 80)

## Key Findings\n\nThis analysis provides comprehensive volatility metrics across multiple US stocks:\n\n1. **Historical Volatility (HV)**: Measures price fluctuations using standard deviation of returns\n2. **Parkinson Volatility**: More efficient estimator using high-low range\n3. **ATR**: Average True Range measures average price movement\n\n## Next Steps\n- Implement Garman-Klass volatility estimator\n- Add volatility forecasting models (GARCH)\n- Compare with implied volatility from options\n- Create volatility trading signals